# Polars SQL encore: same files, another engine
**Additional exercise.** Query the same 2025 European flights using Polars SQL in a Pure Python notebook.
Attach `lh_pond` and run `0. Setup` with `DATA_SCOPE = "one_year"` first. This notebook starts its own SQL context and can run independently of notebooks 1-4.
We will count flights, check coverage, join the airport CSV, and collect a small result. SQL is the main interface; Python registers file scans and displays results.

## 0. Upgrade Polars before importing it
Run this cell first in a fresh session. It upgrades to the latest compatible Polars 2.x release.
If Polars was already imported, restart the Python session after installation, then continue below. For reproducible results, pin the version printed by the next cell in your Fabric environment.

In [ ]:
%pip install --upgrade "polars>=2.0,<3"

### Confirm the version
Polars 2.0 makes streaming the default for lazy queries. Streaming does not guarantee every query fits in memory: joins, sorts, and final results still need resources.

In [ ]:
import polars as pl
from pathlib import Path

print("Polars:", pl.__version__)
if int(pl.__version__.split(".")[0]) < 2:
    raise RuntimeError("Polars 2.0+ is required. Restart the session after upgrading.")

## 1. Register lazy file scans as SQL tables
`scan_parquet` and `scan_csv` describe file sources without loading all rows into a DataFrame.
`SQLContext` gives those sources table names. `.execute()` builds a lazy query; `.collect()` runs it.
The one-year folder keeps the data scope identical to the DuckDB demo. This scan expects the setup dataset's consistent monthly schemas; it does not reproduce DuckDB's `union_by_name` schema reconciliation.

In [ ]:
FLIGHT_FILES = "/lakehouse/default/Files/flights_eu/one_year/flight_list_*.parquet"
AIRPORTS = "/lakehouse/default/Files/flights_eu/reference/airports.csv"

if not list(Path(FLIGHT_FILES).parent.glob("flight_list_*.parquet")):
    raise FileNotFoundError("Attach lh_pond and run 0. Setup with DATA_SCOPE = 'one_year'.")
if not Path(AIRPORTS).is_file():
    raise FileNotFoundError(f"Airport reference file missing: {AIRPORTS}")

flights = pl.scan_parquet(FLIGHT_FILES)
airports = pl.scan_csv(
    AIRPORTS,
    schema_overrides={"ident": pl.String, "municipality": pl.String, "name": pl.String},
).select("ident", "municipality", "name")
sql = pl.SQLContext(flights=flights, airports=airports, eager=False)
print("Registered:", sql.tables())

## 2. Count the same flight rows
The familiar SQL stays. Collect only the one-row aggregate result, not all flight records.
For the standard 2025 setup, expect about 16.2 million rows; use the actual result as your source of truth.

In [ ]:
flight_count = sql.execute("SELECT count(*) AS flights FROM flights").collect()
display(flight_count)

## 3. Inspect the schema and coverage
`collect_schema()` resolves column names and types, potentially reading metadata. It does not collect every data row.
Then SQL checks non-NULL coverage. Empty strings still count as populated.

In [ ]:
print(flights.collect_schema())
display(sql.execute("""
    SELECT count(*) AS rows_total,
           round(100.0 * count(adep) / NULLIF(count(*), 0), 1) AS origin_present_pct,
           round(100.0 * count(ades) / NULLIF(count(*), 0), 1) AS destination_present_pct,
           round(100.0 * count(typecode) / NULLIF(count(*), 0), 1) AS typecode_present_pct,
           round(100.0 * count(registration) / NULLIF(count(*), 0), 1) AS registration_present_pct
    FROM flights
""").collect())

## 4. Name Barcelona's busiest routes
The same two left joins add airport names from CSV to flight Parquet. Missing reference matches remain in the result.
We add airport codes to the sort order so tied counts produce a repeatable top ten in both engines.

In [ ]:
ROUTES_SQL = """
    WITH named_routes AS (
        SELECT f.adep,
               coalesce(dep.municipality, dep.name) AS origin,
               f.ades,
               coalesce(arr.municipality, arr.name) AS destination
        FROM flights AS f
        LEFT JOIN airports AS dep ON f.adep = dep.ident
        LEFT JOIN airports AS arr ON f.ades = arr.ident
        WHERE (f.adep = 'LEBL' OR f.ades = 'LEBL')
          AND f.adep IS NOT NULL
          AND f.ades IS NOT NULL
    )
    SELECT adep, origin, ades, destination, count(*) AS flights
    FROM named_routes
    GROUP BY adep, origin, ades, destination
    ORDER BY flights DESC, adep, ades
    LIMIT 10
"""

barcelona_routes_query = sql.execute(ROUTES_SQL)
barcelona_routes = barcelona_routes_query.collect()
display(barcelona_routes)

## 5. See what Polars plans to do
`explain()` displays the optimized logical plan without executing the result again.
Look for column projection and filters near the scans. SQL is translated into a Polars lazy plan; this is Polars executing the query.
Open Fabric's **Resource usage** panel while collecting results to inspect CPU and memory. This notebook is not a speed benchmark.

In [ ]:
print(barcelona_routes_query.explain())

## 6. Bring back the top 25 airports
The join and aggregation run in Polars. Only the small result is collected into a Polars DataFrame.
The inner join includes airports found in the reference data. Airport IDs keep identically named airports separate.

In [ ]:
AIRPORTS_SQL = """
    SELECT a.ident, a.municipality, a.name, count(*) AS departures
    FROM flights AS f
    JOIN airports AS a ON f.adep = a.ident
    WHERE f.adep IS NOT NULL
    GROUP BY a.ident, a.municipality, a.name
    ORDER BY departures DESC, a.ident
    LIMIT 25
"""
airport_departures = sql.execute(AIRPORTS_SQL).collect()
display(airport_departures)

## 7. Optional: verify the same answers with DuckDB
Set the flag to True to independently read the same files and compare row counts, Barcelona routes, and busiest airports.
This compares values and row order, not engine-specific integer types or timing. No tables or files are written.

In [ ]:
RUN_DUCKDB_CHECK = True

if RUN_DUCKDB_CHECK:
    import duckdb

    with duckdb.connect() as con:
        flight_path = FLIGHT_FILES.replace("'", "''")
        airport_path = AIRPORTS.replace("'", "''")
        con.execute(f"CREATE VIEW flights AS SELECT * FROM read_parquet('{flight_path}', union_by_name=true)")
        con.execute(f"CREATE VIEW airports AS SELECT ident, municipality, name FROM read_csv_auto('{airport_path}')")
        assert con.sql("SELECT count(*) FROM flights").fetchone()[0] == flight_count.item()
        assert con.sql(ROUTES_SQL).fetchall() == barcelona_routes.rows(), "Barcelona route results differ"
        assert con.sql(AIRPORTS_SQL).fetchall() == airport_departures.rows(), "Airport results differ"
    print("PASS: row count, Barcelona routes, and top 25 airports match DuckDB.")
else:
    print("DuckDB comparison skipped. Set RUN_DUCKDB_CHECK = True to run it.")

## Takeaway
Same Lakehouse files, familiar SQL, another engine on one Python node.
DuckDB queries file functions directly; here we registered lazy scans and collected only the results. SQL dialects differ, so validate answers when moving a query between engines.
Polars SQL and the expression API share the lazy engine. You can mix them as a workflow grows.

References: [Polars SQL](https://docs.pola.rs/user-guide/sql/intro/) | [Polars 2.0 migration guide](https://docs.pola.rs/releases/upgrade/2/)